1. Загрузите файл classification.csv. В нем записаны истинные классы объектов выборки (колонка true) и ответы некоторого классификатора (колонка predicted).

In [1]:
import pandas as pd



df1 = pd.read_csv('classification.csv')
y_true1 = df1['true']
y_pred1 = df1['pred']

2. Заполните таблицу ошибок классификации

In [2]:
TP = ((y_true1 == 1) & (y_pred1 == 1)).sum()
FP = ((y_true1 == 0) & (y_pred1 == 1)).sum()
FN = ((y_true1 == 1) & (y_pred1 == 0)).sum()
TN = ((y_true1 == 0) & (y_pred1 == 0)).sum()

print(f'TP FP FN TN: {TP} {FP} {FN} {TN}')

TP FP FN TN: 43 34 59 64


3. Посчитайте основные метрики качества классификатора:

Accuracy (доля верно угаданных) — sklearn.metrics.accuracy

Precision (точность) — sklearn.metrics.accuracy.precision_score

Recall (полнота) — sklearn.metrics.recall_score

F-мера — sklearn.metrics.f1_score

In [6]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, precision_recall_curve)
acc = accuracy_score(y_true1, y_pred1)
prec = precision_score(y_true1, y_pred1)
rec = recall_score(y_true1, y_pred1)
f1 = f1_score(y_true1, y_pred1)

print(f'Accuracy: {round(acc,2)}, Precision: {round(prec, 2)}, Recall: {round(rec,2)}, F1: {round(f1,2)}')

Accuracy: 0.54, Precision: 0.56, Recall: 0.42, F1: 0.48


4. Имеется четыре обученных классификатора.Загрузите этот файл.

In [7]:
df2 = pd.read_csv('scores.csv')
y_true2 = df2['true']
score_cols = ['score_logreg', 'score_svm', 'score_knn', 'score_tree']

5. Посчитайте площадь под ROC-кривой для каждого классификатора. Какой классификатор имеет наибольшее значение метрики AUC-ROC (укажите название столбца с ответами этого классификатора)? Воспользуйтесь функцией sklearn.metrics.roc_auc_score.

In [13]:
auc_values = {}
for col in score_cols:
    auc = roc_auc_score(y_true2, df2[col])
    auc_values[col] = auc
    print(f'{col}: AUC-ROC = {round(auc,2)}') 
auc_values = pd.Series(auc_values)
best_auc_col = auc_values.idxmax()
print(f'Лучший AUC-ROC: {best_auc_col}')


score_logreg: AUC-ROC = 0.72
score_svm: AUC-ROC = 0.71
score_knn: AUC-ROC = 0.64
score_tree: AUC-ROC = 0.69
Лучший AUC-ROC: score_logreg


6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)? Какое значение точности при этом получается?

In [18]:

best_prec_value = -1.0
best_prec_col = None

for col in score_cols:
    precision, recall, thresholds = precision_recall_curve(y_true2, df2[col])
    mask = (recall >= 0.70)
    if mask.any():
        max_prec = precision[mask].max()
        print(f'{col}: max Precision = {round(max_prec,2)} при Recall >= 0.70')
        if max_prec > best_prec_value:
            best_prec_value = max_prec
            best_prec_col = col
    else:
        print(f'{col}: нет Recall >= 0.70')

print(f'Лучший Precision (Recall>=0.7): {best_prec_col} со значением {round(best_prec_value, 2)}')

score_logreg: max Precision = 0.63 при Recall >= 0.70
score_svm: max Precision = 0.62 при Recall >= 0.70
score_knn: max Precision = 0.61 при Recall >= 0.70
score_tree: max Precision = 0.65 при Recall >= 0.70
Лучший Precision (Recall>=0.7): score_tree со значением 0.65
